<a href="https://colab.research.google.com/github/fellipediniz/Analise-de-acoes/blob/main/Plotagem_de_dados_de_a%C3%A7%C3%B5es_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install yfinance

In [7]:
import pandas as pd
import yfinance as yf
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt


In [10]:
# 1. Coleta de dados OHLC
acao = input("Digite o ticker da ação (ex: PETR4, VALE3): ")
ticker = acao.upper()
dataInicio = input(
    "Informe a primeira data para a obtenção dos dados OHLC (formato: AAAA-MM-DD):"
)
dataFim = input(
    "Informe a última data para a obtenção dos dados OHLC (formato: AAAA-MM-DD):"
)

# Download dos dados
dados = yf.download(ticker + ".SA", start=dataInicio, end=dataFim)

# 🔥 CORREÇÃO AQUI: Se as colunas forem um MultiIndex, nós removemos o nível do Ticker
if isinstance(dados.columns, pd.MultiIndex):
    dados.columns = dados.columns.get_level_values(0)

# Remove espaços em branco extras que o yfinance possa ter gerado nas colunas
dados.columns = dados.columns.str.strip()

display(dados)

resposta = input("Deseja plotar os dados? (s/n) ")
resposta = resposta.lower()

if resposta == "s":
    respostaPlot = input("Deseja plotar OHLC completo? (s/n) ")
    respostaPlot = respostaPlot.lower()

    # Tratamento dos dados para o Plotly (Garantindo que a data seja uma coluna acessível)
    dados_plot = dados.reset_index()

    # Garante que a coluna de data não venha com sub-níveis de nome
    if isinstance(dados_plot.columns, pd.MultiIndex):
        dados_plot.columns = dados_plot.columns.get_level_values(0)

    # Mapeamento das escolhas de coluna
    coluna_selecionada = None

    if respostaPlot == "s":
        # Se for o OHLC completo, selecionamos as principais métricas de preço
        colunas_analise = ["Open", "High", "Low", "Close"]
    else:
        escolha = input(
            "Qual cotação deseja plotar? (o = Abertura, h = Mais alta, l = Mais baixa, c = Fechamento, p = Preço ajustado, s = Sair): "
        )
        mapeamento = {
            "o": ["Open"],
            "h": ["High"],
            "l": ["Low"],
            "c": ["Close"],
            "p": ["Adj Close"],
        }

        if escolha == "s":
            print("Ok! Fim do Programa")
            colunas_analise = None
        elif escolha in mapeamento:
            colunas_analise = mapeamento[escolha]
        else:
            print("Por favor digite uma letra válida!")
            colunas_analise = None

    # Se uma coluna válida foi escolhida, prosseguimos para a escolha do tipo de GRÁFICO
    if colunas_analise:
        print("\n--- MENU DE ESTILOS DE PLOTAGEM ---")
        print("1 - Linhas")
        print("2 - Linhas e Pontos")
        print("3 - Barras")
        print("4 - Tabela Pivot (Heatmap por Mês/Ano)")
        tipo_grafico = input("Escolha o tipo de gráfico (1/2/3/4): ")

        # Configuração do título e base visual
        titulo = f"Análise de Cotações para {ticker}.SA - {colunas_analise}"

        # Execução do tipo de gráfico selecionado
        if tipo_grafico == "1":  # LINHAS
            fig = px.line(
                dados_plot,
                x="Date",
                y=colunas_analise,
                title=titulo,
                markers=False,
            )
            fig.show()

        elif tipo_grafico == "2":  # LINHAS E PONTOS
            fig = px.line(
                dados_plot,
                x="Date",
                y=colunas_analise,
                title=titulo,
                markers=True,
            )
            fig.show()

        elif tipo_grafico == "3":  # BARRAS
            fig = px.bar(
                dados_plot, x="Date", y=colunas_analise, title=titulo
            )
            fig.show()

        elif tipo_grafico == "4":  # PIVOT
            # Engenharia de Recursos (Feature Engineering): Isolando Ano e Mês para Pivotar mercado financeiro
            dados_plot["Ano"] = dados_plot["Date"].dt.year
            dados_plot["Mês"] = dados_plot["Date"].dt.strftime("%m-%B")

            # Caso o usuário tenha escolhido o OHLC completo, para o pivot usamos apenas o Fechamento ('Close') como referência
            coluna_pivot = (
                colunas_analise[0]
                if len(colunas_analise) == 1
                else "Close"
            )

            # Criando a tabela pivô consolidando pela média do preço no mês
            df_pivot = dados_plot.pivot_table(
                index="Ano",
                columns="Mês",
                values=coluna_pivot,
                aggfunc="mean",
            )

            # Plotando a tabela pivô como um Heatmap
            fig = px.imshow(
                df_pivot,
                labels=dict(x="Mês", y="Ano", color="Preço Médio"),
                x=df_pivot.columns,
                y=df_pivot.index,
                text_auto=".2f",
                title=f"Tabela Pivô (Média de {coluna_pivot} por Ano/Mês) - {ticker}",
                color_continuous_scale="RdYlGn",  # Verde para preços altos, vermelho para baixos
            )
            fig.show()
        else:
            print("Opção de gráfico inválida.")

else:
    print("Ok! Fim do Programa")


Digite o ticker da ação (ex: PETR4, VALE3): bbas3
Informe a primeira data para a obtenção dos dados OHLC (formato: AAAA-MM-DD):2026-06-01
Informe a última data para a obtenção dos dados OHLC (formato: AAAA-MM-DD):2026-06-16


/tmp/ipykernel_10513/3032166519.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dados = yf.download(ticker + ".SA", start=dataInicio, end=dataFim)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2026-06-01,19.938738,20.405431,19.908948,20.365713,50279700
2026-06-02,19.889999,20.180000,19.889999,20.139999,19384600
2026-06-03,19.530001,19.870001,19.459999,19.770000,27323000
2026-06-05,19.170000,19.650000,19.170000,19.600000,51227600
2026-06-08,19.100000,19.340000,19.100000,19.170000,15329800
2026-06-09,19.110001,19.469999,19.040001,19.230000,18242900
2026-06-10,19.000000,19.180000,18.870001,19.120001,18849900
2026-06-11,19.410000,19.559999,18.910000,19.070000,32600200
2026-06-12,19.459999,19.660000,19.219999,19.299999,13773000


Deseja plotar os dados? (s/n) s
Deseja plotar OHLC completo? (s/n) s

--- MENU DE ESTILOS DE PLOTAGEM ---
1 - Linhas
2 - Linhas e Pontos
3 - Barras
4 - Tabela Pivot (Heatmap por Mês/Ano)
Escolha o tipo de gráfico (1/2/3/4): 2
